<a href="https://colab.research.google.com/github/Yahir-7/Data_Center_GeoPandas/blob/main/all_data_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# install packages if needed
# !pip install geopandas pandas openpyxl pyogrio -q

import pandas as pd
import geopandas as gpd
import glob
from google.colab import files

# load county boundaries
counties = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_20m.zip"
).to_crs(epsg=4326)

counties["FIPS"] = counties["GEOID"].astype(str).str.zfill(5)

remove_statefps = ["15", "60", "66", "69", "78"]
counties = counties[~counties["STATEFP"].isin(remove_statefps)].copy()

master = counties[
    ["FIPS", "STATEFP", "STATE_NAME", "NAME", "NAMELSAD", "ALAND", "geometry"]
].copy()

master = master.rename(columns={"NAME": "County"})

# find local files
def find_file(filename):
    matches = glob.glob(f"**/{filename}", recursive=True)
    return matches[0] if len(matches) > 0 else None

def find_file_contains(text):
    matches = glob.glob(f"**/*{text}*", recursive=True)
    return matches[0] if len(matches) > 0 else None

def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("$", "", regex=False),
        errors="coerce"
    )

# add extreme heat data
heat_path = find_file("EJScreen_HI.shp")

if heat_path is None:
    raise FileNotFoundError("EJScreen_HI.shp was not found.")

heat = gpd.read_file(heat_path).to_crs(epsg=4326)

heat_col = "Average_Da"

heat_points = heat.to_crs(epsg=5070).copy()
heat_points["geometry"] = heat_points.geometry.centroid
heat_points = heat_points.to_crs(epsg=4326)

heat_joined = gpd.sjoin(
    heat_points[[heat_col, "geometry"]],
    master[["FIPS", "geometry"]],
    how="inner",
    predicate="within"
)

heat_county = heat_joined.groupby("FIPS")[heat_col].mean().reset_index()
heat_county = heat_county.rename(columns={heat_col: "extreme_heat"})

master = master.merge(heat_county, on="FIPS", how="left")

# add drought data
drought_path = find_file("Census_Tract.shp")

if drought_path is None:
    raise FileNotFoundError("Census_Tract.shp was not found.")

drought = gpd.read_file(drought_path).to_crs(epsg=4326)

drought_col = "DRGT_AFREQ"

drought_points = drought.to_crs(epsg=5070).copy()
drought_points["geometry"] = drought_points.geometry.centroid
drought_points = drought_points.to_crs(epsg=4326)

drought_joined = gpd.sjoin(
    drought_points[[drought_col, "geometry"]],
    master[["FIPS", "geometry"]],
    how="inner",
    predicate="within"
)

drought_county = drought_joined.groupby("FIPS")[drought_col].mean().reset_index()
drought_county = drought_county.rename(columns={drought_col: "drought_severity"})

master = master.merge(drought_county, on="FIPS", how="left")

# add county health rankings data
chr_file = find_file("2025 County Health Rankings Data - v4.xlsx")

if chr_file is None:
    chr_file = find_file_contains("2025 County Health Rankings Data")

if chr_file is None:
    raise FileNotFoundError("County Health Rankings Excel file was not found.")

chr_data = pd.read_excel(
    chr_file,
    sheet_name="Select Measure Data",
    header=1
)

chr_data = chr_data[chr_data["County"].notna()].copy()

chr_data["FIPS"] = (
    chr_data["FIPS"]
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

chr_columns = {
    "Average Daily PM2.5": "average_daily_pm25",
    "Presence of Water Violation": "presence_of_water_violation",
    "% Severe Housing Problems": "percent_severe_housing_problems",
    "% Households with Broadband Access": "percent_households_broadband",
    "% Unemployed": "percent_unemployed",
    "Income Ratio": "income_inequality_ratio"
}

keep_cols = ["FIPS"]

for old_col in chr_columns:
    if old_col in chr_data.columns:
        keep_cols.append(old_col)

chr_clean = chr_data[keep_cols].copy()
chr_clean = chr_clean.rename(columns=chr_columns)

master = master.merge(chr_clean, on="FIPS", how="left")

# add LEAD energy data
lead_file = find_file_contains("LEAD Tool Data Census Tracts")

if lead_file is None:
    raise FileNotFoundError("LEAD Tool Data Census Tracts CSV file was not found.")

lead = pd.read_csv(
    lead_file,
    skiprows=8
)

lead["FIPS"] = (
    lead["Geography ID"]
    .astype(str)
    .str.zfill(11)
    .str[:5]
)

lead["Total Households"] = clean_numeric(lead["Total Households"])

lead["Avg. Annual Energy Cost ($) (Electricity)"] = clean_numeric(
    lead["Avg. Annual Energy Cost ($) (Electricity)"]
)

lead["Avg. Annual Energy Cost ($) (Gas)"] = clean_numeric(
    lead["Avg. Annual Energy Cost ($) (Gas)"]
)

lead = lead[lead["Total Households"].notna()].copy()

lead["electricity_weighted"] = (
    lead["Avg. Annual Energy Cost ($) (Electricity)"]
    * lead["Total Households"]
)

lead["gas_weighted"] = (
    lead["Avg. Annual Energy Cost ($) (Gas)"]
    * lead["Total Households"]
)

lead_county = lead.groupby("FIPS").agg(
    electricity_weighted=("electricity_weighted", "sum"),
    gas_weighted=("gas_weighted", "sum"),
    total_households_lead=("Total Households", "sum")
).reset_index()

lead_county["electricity_price"] = (
    lead_county["electricity_weighted"]
    / lead_county["total_households_lead"]
)

lead_county["gas_price"] = (
    lead_county["gas_weighted"]
    / lead_county["total_households_lead"]
)

lead_county = lead_county[
    [
        "FIPS",
        "electricity_price",
        "gas_price"
    ]
].copy()

master = master.merge(
    lead_county,
    on="FIPS",
    how="left"
)

# add census demographic data
dp05_file = find_file_contains("ACSDP5Y2023.DP05")

if dp05_file is None:
    raise FileNotFoundError("ACSDP5Y2023.DP05 Excel file was not found.")

acs_data = pd.read_excel(
    dp05_file,
    sheet_name="Data",
    header=None
)

total_pop_row = acs_data[acs_data[0] == "Total population"].index[0]
under18_row = acs_data[acs_data[0] == "Under 18 years"].index[0]
older65_row = acs_data[acs_data[0] == "65 years and over"].index[0]

white_alone_rows = acs_data[acs_data[0] == "White alone"].index
non_hispanic_white_row = white_alone_rows[-1]

county_columns = list(range(5, acs_data.shape[1], 4))

demo_data = []

for col in county_columns:
    county_state = acs_data.iloc[0, col]

    if pd.isna(county_state):
        continue

    county_state = str(county_state)

    if "," not in county_state:
        continue

    demo_data.append({
        "county_state": county_state,
        "total_population": acs_data.iloc[total_pop_row, col],
        "percent_children_under_18": acs_data.iloc[under18_row, col + 2],
        "percent_older_adults_65_plus": acs_data.iloc[older65_row, col + 2],
        "percent_non_hispanic_white": acs_data.iloc[non_hispanic_white_row, col + 2]
    })

demo = pd.DataFrame(demo_data)

demo[["county", "state"]] = demo["county_state"].str.rsplit(
    ", ",
    n=1,
    expand=True
)

demo["total_population"] = clean_numeric(demo["total_population"])
demo["percent_children_under_18"] = clean_numeric(demo["percent_children_under_18"])
demo["percent_older_adults_65_plus"] = clean_numeric(demo["percent_older_adults_65_plus"])
demo["percent_non_hispanic_white"] = clean_numeric(demo["percent_non_hispanic_white"])

demo["percent_people_of_color"] = (
    100 - demo["percent_non_hispanic_white"]
)

demo_clean = demo[
    [
        "county",
        "state",
        "total_population",
        "percent_children_under_18",
        "percent_older_adults_65_plus",
        "percent_people_of_color"
    ]
].copy()

master = master.merge(
    demo_clean,
    left_on=["NAMELSAD", "STATE_NAME"],
    right_on=["county", "state"],
    how="left"
)

# calculate population density
master["area_sq_miles"] = master["ALAND"] / 2589988.110336

master["population_density"] = (
    master["total_population"] / master["area_sq_miles"]
)

# save and download csv
final_csv = master.drop(
    columns=["geometry", "ALAND", "NAMELSAD", "county", "state"],
    errors="ignore"
).copy()

needed_columns = [
    "drought_severity",
    "extreme_heat",
    "electricity_price",
    "gas_price",
    "percent_households_broadband",
    "percent_unemployed",
    "income_inequality_ratio",
    "percent_severe_housing_problems",
    "average_daily_pm25",
    "presence_of_water_violation",
    "total_population",
    "population_density",
    "percent_children_under_18",
    "percent_older_adults_65_plus",
    "percent_people_of_color"
]

missing_columns = [col for col in needed_columns if col not in final_csv.columns]

if len(missing_columns) > 0:
    raise ValueError(f"These columns are still missing: {missing_columns}")

output_file = "yahir_geopandas_all-datasets.csv"

final_csv.to_csv(output_file, index=False)

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>